In [1]:
import os
import sys

project_root = os.path.abspath("..")   # if the notebook is inside notebooks/
sys.path.insert(0, project_root)
print(project_root)

c:\Projects\Service Event Framework\Improvement\code


In [2]:
import os
import numpy as np
import pandas as pd
from utils.dbconnection import engine
from utils.config import config

2026-09-24 12:09:26,570 - Logger: Log file: c:\Projects\Service Event Framework\Improvement\code\logs\SEF_Improvement_20260924_120926.log
2026-09-24 12:09:26,610 - SQLAlchemy: Database connection configured successfully!


In [3]:
from utils.config import config

In [4]:
file_name = 'ServiceEvent_Material.sql'
sql_path = os.path.join(config.locations.queries_folder, file_name)
df = pd.read_sql_query(open(sql_path, "r").read(), engine)
df.head()

,DServiceEventID,Serviceeventcode,ServiceEventName,AND_OR,Priority,confidence,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription
0,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10.0,107033,BLADE BEARING ø54,31,"Bearings, Blade TRAC"
1,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10.0,760306,BLADE BEARING MODULE,327,Nacelle modules and
2,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10.0,76221907,BLADE BEARING MODULE,30,Bearings
3,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10.0,76221971,BLADE BEARING MODULE,327,Nacelle modules and
4,4,GenBearReplace,Generator Bearing Replacement,OR,7.8,10.0,103566,"BEARING,BALL,6344 MC3,220 mm",30,Bearings


In [5]:
file_name = 'ServiceEvent_MaterialGroup.sql'
sql_path = os.path.join(config.locations.queries_folder, file_name)
qw = pd.read_sql_query(open(sql_path, "r").read(), engine)
qw.head()

,DServiceEventID,Serviceeventcode,ServiceEventName,AND_OR,Priority,confidence,MaterialGroup,MaterialGroupDescription
0,262,SkyLightCoverReplace,Sky Light Cover replacement,OR,5.16,5.0,105,Covers
1,3,GearReplace,Gearbox,AND,10.00,10.0,211,Gearboxes not mounte
2,631,BladeStudAssemblyReplace,Blade Stud Assembly Replace,AND,7.07,6.0,467,Site Parts Blade/Hub
3,6,GenReplace,Generator Replacement,AND,9.10,10.0,221,Generators not mount
4,12,YawGearReplace,Yaw Gear Replacement,OR,7.47,7.0,218,"Gears, yaw TRACE"


1. Materials being present in multiple events

In [6]:
df1 = df[['Serviceeventcode', 'Itemnumber']].drop_duplicates()
df1.shape

(8109, 2)

In [7]:
df1c = pd.DataFrame(df1['Itemnumber'].value_counts()).reset_index()
df1c = df1c.loc[df1c['count']>1]
df1c.shape

(27, 2)

In [8]:
df1c['Itemnumber'].unique()

<ArrowStringArray>
['29023724', '29097212',   '114447',   '753256',   '753257',   '753264',
   '753770',   '753780',   '753889', '29020336',   '153828', '29080472',
 '29087200', '29133149', '29133175', '29153440', '29153441', '29332819',
 '29332860', '29333356', '29333364', '29333367', '29333368', '29349561',
 '29349658', '29452248', '29204214']
Length: 27, dtype: str

In [9]:
df1m = df1.merge(df1c, on='Itemnumber', how='inner')
df1m.sort_values(by='Itemnumber', inplace=True)
# Merge Material names to the output
df1m = df1m.merge(df[['Itemnumber','MaterialName']], on='Itemnumber', how='left')
# Merge AND_OR column to the output
df1m = df1m.merge(df[['Serviceeventcode','AND_OR']], on='Serviceeventcode', how='left')
df1m.head()

,Serviceeventcode,Itemnumber,count,MaterialName,AND_OR
0,ThermoValveCACReplace,114447,2,"THERMOSTAT,0 °C,60 °C,NO",OR
1,ThermoValveCACReplace,114447,2,"THERMOSTAT,0 °C,60 °C,NO",OR
2,ThermoValveCACReplace,114447,2,"THERMOSTAT,0 °C,60 °C,NO",OR
3,ThermoValveCACReplace,114447,2,"THERMOSTAT,0 °C,60 °C,NO",OR
4,ThermoValveCACReplace,114447,2,"THERMOSTAT,0 °C,60 °C,NO",OR


2. Material Groups present in multiple events

In [10]:
qw1 = qw[['Serviceeventcode', 'MaterialGroup', 'AND_OR']]
qw1 = qw1.drop_duplicates()
qw1.shape

(47, 3)

In [11]:
qw1c = pd.DataFrame(qw1['MaterialGroup'].value_counts()).reset_index()
qw1c = qw1c.loc[qw1c['count']>1]
qw1c.shape, qw1c['MaterialGroup'].unique()

((4, 2),
 <ArrowStringArray>
 ['75', '255', '523', '490']
 Length: 4, dtype: str)

In [12]:
qw1m = qw1.merge(qw1c, on='MaterialGroup', how='inner')
qw1m.sort_values(by='MaterialGroup', inplace=True)
# Merge Material group names to the output
qw1m = qw1m.merge(qw[['MaterialGroup','MaterialGroupDescription']], on='MaterialGroup', how='left')
qw1m.head()

,Serviceeventcode,MaterialGroup,AND_OR,count,MaterialGroupDescription
0,RTMHosesReplace,255,AND,2,Hydr.-lubr. & cool.
1,RTMHosesReplace,255,AND,2,Hydr.-lubr. & cool.
2,HydCoolPipeReplace,255,AND,2,Hydr.-lubr. & cool.
3,HydCoolPipeReplace,255,AND,2,Hydr.-lubr. & cool.
4,NoseRepairKit_mat,490,AND,2,Spinners / nose cone


3. Material being present in one service event but the respective material group in a different service event

In [13]:
req_cols = ['Serviceeventcode', 'Itemnumber', 'MaterialName', 'MaterialGroup', 'MaterialGroupDescription','AND_OR']
mg_all  = df[req_cols].drop_duplicates()
mg_all.shape

(8132, 6)

In [14]:
mg1 = mg_all.merge(qw1, on='MaterialGroup', how='inner', suffixes=('_primary', '_secondary'))
mg1

,Serviceeventcode_primary,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription,AND_OR_primary,Serviceeventcode_secondary,AND_OR_secondary
0,BladeBearingReplace,107033,BLADE BEARING ø54,31,"Bearings, Blade TRAC",OR,BladeBearingReplace,OR
1,MainBearingReplace,29155641,MAIN SHAFT SER. ASSY 4MW MK 3E,455,"Shafts, main",OR,MainBearingReplace,OR
2,MainBearingReplace,29195008,PTR & AC GEN 5.6MW LTQ,540,Transmission modules,OR,PowerTrainAssembly_mat,AND
3,PitchCylinderReplace,108613,"CYLINDER,HYDRAULIC,125 mm,80 mm,922 mm",251,"Hy cylinders, pitch",OR,PitchCylinderReplace,OR
4,PitchCylinderReplace,108614,"CYLINDER,HYDRAULIC,125 mm,80 mm,922 mm",251,"Hy cylinders, pitch",OR,PitchCylinderReplace,OR
...,...,...,...,...,...,...,...,...
1675,HydOilHoseReplace,781673,"HOSE ASSY,HYD,9.525 mm,2325 mm,330 bar",254,Hydraulic hoses,OR,HydOilHoseReplace,OR
1676,HydOilHoseReplace,781674,"HOSE ASSY,HYD,6.35 mm,2105 mm,400 bar",254,Hydraulic hoses,OR,HydOilHoseReplace,OR
1677,HydOilHoseReplace,782038,"HOSE ASSY,HYD,19.05 mm,3760 mm,248 bar",254,Hydraulic hoses,OR,HydOilHoseReplace,OR
1678,HydOilHoseReplace,788931,"HOSE ASSY,HYD,12.7 mm,1580 mm,275 bar",254,Hydraulic hoses,OR,HydOilHoseReplace,OR


In [15]:
mismatches = mg1.loc[mg1['Serviceeventcode_primary'] != mg1['Serviceeventcode_secondary']]
mismatches.head()

,Serviceeventcode_primary,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription,AND_OR_primary,Serviceeventcode_secondary,AND_OR_secondary
2,MainBearingReplace,29195008,PTR & AC GEN 5.6MW LTQ,540,Transmission modules,OR,PowerTrainAssembly_mat,AND
24,GearHSS,29102035,ADA REPAIR KIT ZFR916011454 GE,383,Power module,OR,PPRSkiipReplace,OR
25,GearHSS,789405,EF901E-404L KIT STAGE 3 - 60Hz,211,Gearboxes not mounte,OR,GearReplace,AND
26,GenFanReplace,70000506,COOLING FAN - ICDS,230,Heat exchangers,OR,CACHeatExchangerMGReplace,OR
27,CACPumpReplace,101011,COOLING UNIT V39 OIL COOLER,230,Heat exchangers,OR,CACHeatExchangerMGReplace,OR


In [16]:
# validate mismatches: Case 1
df.loc[df['Itemnumber']=='29195008']

,DServiceEventID,Serviceeventcode,ServiceEventName,AND_OR,Priority,confidence,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription
86,7,MainBearingReplace,Main Bearing Arrangement Replacement,OR,9.0,10.0,29195008,PTR & AC GEN 5.6MW LTQ,540,Transmission modules


In [17]:
qw.loc[qw['MaterialGroup']=='540']

,DServiceEventID,Serviceeventcode,ServiceEventName,AND_OR,Priority,confidence,MaterialGroup,MaterialGroupDescription
6,956,PowerTrainAssembly_mat,Power Train Assembly Replace,AND,9.2,10.0,540,Transmission modules


In [18]:
# validate mismatches: Case 2
df.loc[df['Itemnumber']=='29102035']

,DServiceEventID,Serviceeventcode,ServiceEventName,AND_OR,Priority,confidence,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription
358,18,GearHSS,GearBox HSS Stage,OR,7.72,10.0,29102035,ADA REPAIR KIT ZFR916011454 GE,383,Power module


In [19]:
qw.loc[qw['MaterialGroup']=='383']

,DServiceEventID,Serviceeventcode,ServiceEventName,AND_OR,Priority,confidence,MaterialGroup,MaterialGroupDescription
7,97,PPRSkiipReplace,PPR SKiiPack Replacement,OR,7.03,6.0,383,Power module


4. Materials belonging to same material group are in different Service Events

In [20]:
df.head()

,DServiceEventID,Serviceeventcode,ServiceEventName,AND_OR,Priority,confidence,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription
0,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10.0,107033,BLADE BEARING ø54,31,"Bearings, Blade TRAC"
1,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10.0,760306,BLADE BEARING MODULE,327,Nacelle modules and
2,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10.0,76221907,BLADE BEARING MODULE,30,Bearings
3,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10.0,76221971,BLADE BEARING MODULE,327,Nacelle modules and
4,4,GenBearReplace,Generator Bearing Replacement,OR,7.8,10.0,103566,"BEARING,BALL,6344 MC3,220 mm",30,Bearings


In [21]:
req_cols = ['Serviceeventcode', 'MaterialGroup', 'AND_OR']
dmg_all = df[req_cols].drop_duplicates()
dmg_all.head()

,Serviceeventcode,MaterialGroup,AND_OR
0,BladeBearingReplace,31,OR
1,BladeBearingReplace,327,OR
2,BladeBearingReplace,30,OR
4,GenBearReplace,30,OR
6,GenBearReplace,222,OR


In [22]:
dmgg = dmg_all.groupby('MaterialGroup').size().reset_index(name='count')
dmgg.sort_values(by='count', ascending=False, inplace=True)
dmgg.head()

,MaterialGroup,count
14,175,51
89,445,39
50,275,33
47,255,32
127,65,29


In [37]:
dmgg2 = dmgg.loc[dmgg['count']==2]
dmgg2 = df.loc[df['MaterialGroup'].isin(dmgg2['MaterialGroup'])]
dmgg2 = dmgg2.sort_values(by='MaterialGroup')
req_cols = ['Serviceeventcode', 'MaterialGroup', 'MaterialGroupDescription', 'AND_OR']
dmgg2f = dmgg2[req_cols].drop_duplicates()
dmgg2f.head(10)

,Serviceeventcode,MaterialGroup,MaterialGroupDescription,AND_OR
2437,SkyLightReplace,105,Covers,OR
6778,NacelleBottomCoverRepair,105,Covers,OR
4138,DownConductorReceptorRepair,145,Earthing equipment,OR
7451,BladeLightReceptorScrew_mat,145,Earthing equipment,OR
3069,ControllerCabinetFanReplace,229,Ground controllers,OR
3489,GroundCntrlReplace,229,Ground controllers,OR
2337,PPRHVCableReplace,248,High voltage cables,OR
2807,ConvCableReplace,248,High voltage cables,OR
358,GearHSS,383,Power module,OR
1481,PPRSkiipReplace,383,Power module,OR


In [34]:
def get_n_replicates(dmgg, df, n=2):
    dmgg_n = dmgg.loc[dmgg['count']==n]
    dmgg_n = df.loc[df['MaterialGroup'].isin(dmgg_n['MaterialGroup'])]
    dmgg_n = dmgg_n.sort_values(by='MaterialGroup')
    req_cols = ['Serviceeventcode', 'MaterialGroup', 'MaterialGroupDescription', 'AND_OR']
    dmgg_nf = dmgg_n[req_cols].drop_duplicates()
    return dmgg_nf

In [35]:
get_n_replicates(dmgg, df, 3).shape

(48, 4)

In [38]:
req_cols = ['Serviceeventcode',  'Itemnumber', 'MaterialName', 'MaterialGroup']
mrg_mixie = dmgg2f.merge(df[req_cols], on=['Serviceeventcode','MaterialGroup'], how='left')
mrg_mixie.head(10)


,Serviceeventcode,MaterialGroup,MaterialGroupDescription,AND_OR,Itemnumber,MaterialName
0,SkyLightReplace,105,Covers,OR,10205880,SKYLIGHT
1,NacelleBottomCoverRepair,105,Covers,OR,60112900,NACELLE COVER BOTTOM NM72/82
2,DownConductorReceptorRepair,145,Earthing equipment,OR,731051,"LIGHTNING RECEPTOR SCREW, 35MM"
3,DownConductorReceptorRepair,145,Earthing equipment,OR,731052,"LIGHTNING RECEPTOR SCREW, 19MM"
4,DownConductorReceptorRepair,145,Earthing equipment,OR,731053,LIGHTNING RECEPTOR SCREW 25MM
5,DownConductorReceptorRepair,145,Earthing equipment,OR,731055,"LIGHTNING RECEPTOR, 30MM"
6,DownConductorReceptorRepair,145,Earthing equipment,OR,731056,"LIGHTNING RECEPTOR, 45MM"
7,DownConductorReceptorRepair,145,Earthing equipment,OR,751111,LIGHTNING RECEPTOR
8,DownConductorReceptorRepair,145,Earthing equipment,OR,761085,"LIGHTN.RECEPTOR 56 MM,39 M BL."
9,DownConductorReceptorRepair,145,Earthing equipment,OR,761086,"LIGHTNING RECEPTOR, TILTED"


Save results

In [43]:
mrg_mixie.loc[mrg_mixie['MaterialGroup']=='105']

,Serviceeventcode,MaterialGroup,MaterialGroupDescription,AND_OR,Itemnumber,MaterialName
0,SkyLightReplace,105,Covers,OR,10205880,SKYLIGHT
1,NacelleBottomCoverRepair,105,Covers,OR,60112900,NACELLE COVER BOTTOM NM72/82


In [39]:
df_save_dict = {
    'event_material': df,
    'event_materialGroup': qw,
    'material_duplicacy': df1m,
    'materialGroup_duplicacy': qw1m,
    'material_vs_materialGroup': mismatches,
    'materialgroup_nEvents_list': mrg_mixie,
    'materialgroup_nEvents': dmgg2
}

In [41]:
from pathlib import Path

output_dir = Path(config.locations.outputs)
output_dir.mkdir(parents=True, exist_ok=True)
excel_path = output_dir / "service_event_analysis.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, df_sheet in df_save_dict.items():
        if isinstance(df_sheet, pd.DataFrame):
            df_sheet.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            pd.DataFrame(df_sheet).to_excel(writer, sheet_name=sheet_name, index=False)

excel_path

WindowsPath('C:/Projects/Service Event Framework/Improvement/code/data/outputs/service_event_analysis.xlsx')